# PD-001 development launcher — full retraining currently blocked

The performance repair requires report review first. For now import **performance-notebook.ipynb** and Run All to measure the old and repaired implementations using bounded synthetic inputs. This ordinary development launcher retains the original automatic flow, but full training fails before GPU work until the performance and persistence review binds the new source/config. Old checkpoints must not continue as the repaired implementation's scientific curve.


In [ ]:
"""Standard-library session supervisor, also embedded verbatim before Notebook checkout."""
import json
import os
from pathlib import Path
import signal
import subprocess
import time

SESSION_SECONDS = 7200
SAVE_RESERVE_SECONDS = 600
TERMINATION_RESERVE_SECONDS = 10
DEADLINE_ENV = "PD001_SESSION_DEADLINE_MONOTONIC"


def session_deadline():
    now = time.monotonic()
    requested = float(os.environ.get(DEADLINE_ENV, now + SESSION_SECONDS))
    if not now < requested < float("inf"):
        raise TimeoutError("PD-001 session budget exhausted; preserve Output")
    deadline = min(requested, now + SESSION_SECONDS)
    os.environ[DEADLINE_ENV] = str(deadline)
    return deadline


def pause_due():
    value = os.environ.get(DEADLINE_ENV)
    return value is not None and time.monotonic() >= float(value) - SAVE_RESERVE_SECONDS


class SessionBudget:
    def __init__(self, status_path):
        self.deadline = session_deadline()
        self.status_path = Path(status_path)

    def record(self, status):
        self.status_path.parent.mkdir(parents=True, exist_ok=True)
        temporary = self.status_path.with_suffix(".tmp")
        temporary.write_text(json.dumps({
            "status": status, "wall_limit_seconds": SESSION_SECONDS,
            "remaining_seconds": max(0, self.deadline - time.monotonic()),
            "scientific_decision": False,
            "note": "Retain Kaggle Output; only completed checkpoints/evidence can be restored."
        }, indent=2) + "\n")
        temporary.replace(self.status_path)

    def run(self, command, **kwargs):
        remaining = self.deadline - time.monotonic() - TERMINATION_RESERVE_SECONDS
        if remaining <= 0:
            self.record("SESSION_TIME_LIMIT")
            raise TimeoutError("PD-001 two-hour session limit; preserve Output")
        # One group contains the launcher, benchmark workers and training workers.
        # Killing only the immediate launcher can leave CUDA work running.
        process = subprocess.Popen(command, start_new_session=True, **kwargs)
        try:
            code = process.wait(timeout=remaining)
        except BaseException as error:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=3)
            except subprocess.TimeoutExpired:
                pass
            # Group members may outlive an already exited leader.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait(timeout=3)
            self.record("SESSION_TIME_LIMIT" if isinstance(error, subprocess.TimeoutExpired) else "INTERRUPTED")
            raise
        if code:
            self.record("FAILED")
            raise subprocess.CalledProcessError(code, command)
        return code

BUDGET = SessionBudget('/kaggle/working/pd001-session-status.json')
BUDGET.record('RUNNING')
print('Session wall limit: 120 minutes; training checkpoint pause begins with 10 minutes left',flush=True)

from pathlib import Path
import subprocess, sys
MODE = 'auto'  # preflight -> registered experiment; any failure stops
SOURCE_COMMIT = '4927956e529d3401364fa2b9bce70a6c62831c73'
ROOT = Path('/kaggle/working/minicells-pd001-' + SOURCE_COMMIT[:12])
if MODE not in ('auto','audit','params','dry-run','preflight'):
    raise RuntimeError('Choose auto or an engineering diagnostic mode')
if SOURCE_COMMIT.startswith('__'):
    raise RuntimeError('Import the committed notebook with its pinned source')
ROOT.mkdir(parents=True,exist_ok=True)
def git(*args):
    print('Setup: git ' + ' '.join(args),flush=True)
    BUDGET.run(['git',*args],cwd=ROOT)
ready = False
if (ROOT/'.git').exists():
    head = subprocess.run(['git','rev-parse','HEAD'],cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE)
    ready = head.returncode == 0 and head.stdout.strip() == SOURCE_COMMIT and (ROOT/'research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth/run_pd001.py').exists()
if not ready:
    if not (ROOT/'.git').exists():
        git('init')
        git('remote','add','origin','https://github.com/ArcheLabs/mini-cells.git')
    origin = subprocess.check_output(['git','remote','get-url','origin'],cwd=ROOT,text=True).strip()
    if origin != 'https://github.com/ArcheLabs/mini-cells.git':
        raise RuntimeError('Unexpected source checkout origin')
    git('config','remote.origin.promisor','true')
    git('config','remote.origin.partialclonefilter','blob:none')
    git('sparse-checkout','init','--no-cone')
    git('sparse-checkout','set','--no-cone','/pyproject.toml','/src/','/shared/python/',
        '/research/projects/cellular-language-models/experiments/006-path-first-clm/',
        '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/',
        '/tests/research/path_first_pd001/',
        '/research/governance/formal_seed_registry.json',
        '/research/projects/**/config/*.json', '/research/projects/**/configs/*.json',
        '/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/',
        '/research/projects/continual-learning/context/04-continual-learning/scripts/research/')
    git('fetch','--depth=1','--filter=blob:none','origin',SOURCE_COMMIT)
    git('checkout','--detach',SOURCE_COMMIT)


In [ ]:
ENTRY = ROOT/'research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth/run_pd001.py'
if not ENTRY.exists():
    raise RuntimeError('Checkout is incomplete; preserve output, remove only the incomplete code checkout and retry')
print('Automatic preflight, assets, development training and return; failures stop the flow',flush=True)
import importlib.util
missing = [package for package,module in [('datasets','datasets'),('tokenizers','tokenizers'),('huggingface_hub','huggingface_hub')] if importlib.util.find_spec(module) is None]
if missing:
    BUDGET.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*missing])
import os, time
out = Path('/kaggle/working/pd001-development-' + SOURCE_COMMIT[:12])
env = {**os.environ, 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1'}
if MODE != 'auto':
    env['CUDA_VISIBLE_DEVICES'] = ''
config = ENTRY.parent/'config/proposed.json'
command = [sys.executable,str(ENTRY),'--mode',MODE,'--device','cpu','--config',str(config),'--output',str(out)]
if MODE == 'auto':
    command += ['--devices','auto']
BUDGET.run(command,cwd=ROOT,env=env)
print({'status':'AUTOMATIC_FLOW_COMPLETED','output':str(out),'scope':'development_only'},flush=True)

BUDGET.record('COMPLETED')
